В блокноте создается датасет с выровненными лицами 224 * 224 для вычисления identification rate metric, тем же способом, что и датасет для обучения моделей для распознавания лиц. Для query возьмем 100 человек по 3 фото на каждого, а для distractors возьмем 1000 человек по 1 фото.

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from skimage.transform import SimilarityTransform
import cv2

Читаем датафрейм всех фото по всем личностям.

In [2]:
id_df = pd.read_csv('CelebA/Anno/identity_CelebA.txt', header=None, sep='\\s+')
id_df

,0,1
0,000001.jpg,2880
1,000002.jpg,2937
2,000003.jpg,8692
3,000004.jpg,5805
4,000005.jpg,9295
...,...,...
202594,202595.jpg,9761
202595,202596.jpg,7192
202596,202597.jpg,9852
202597,202598.jpg,5570


Находим список личностей, которые использовались для обучения модели распознавания лиц. (Датафрейм для валидации взят, потому что он меньше, чем для трейна, а люди там одинаковые, но с разными фото.)

In [3]:
used_id_df = pd.read_csv('aligned/val_identity.csv', header=None)
used_id_df

,0,1
0,000002.jpg,2937
1,000003.jpg,8692
2,000051.jpg,1446
3,000054.jpg,8443
4,000067.jpg,1175
...,...,...
2995,202383.jpg,4992
2996,202409.jpg,5061
2997,202499.jpg,10157
2998,202539.jpg,6411


In [4]:
used_ids = set(used_id_df[1].values)

Для выбора личностей будем использовать датафрейм с id личности и количеством ее фото.

In [5]:
size_df = id_df.groupby(1).size().reset_index(name='size')
size_df

,1,size
0,1,29
1,2,8
2,3,25
3,4,22
4,5,20
...,...,...
10172,10173,30
10173,10174,30
10174,10175,30
10175,10176,30


Исключим из них тех, которые использовались для обучения модели.

In [6]:
size_df = size_df.drop(size_df[size_df[1].isin(used_ids)].index)
size_df

,1,size
0,1,29
1,2,8
2,3,25
3,4,22
4,5,20
...,...,...
10170,10171,20
10171,10172,11
10172,10173,30
10175,10176,30


Дальше отбираем 100 личностей, у которых есть хотя бы по 3 фото, для query.

In [7]:
query_ids = set(size_df[size_df['size'] >= 3].sample(100)[1].values)

Исключим отобранных.

In [8]:
size_df = size_df.drop(size_df[size_df[1].isin(query_ids)].index)
size_df

,1,size
0,1,29
1,2,8
2,3,25
3,4,22
4,5,20
...,...,...
10170,10171,20
10171,10172,11
10172,10173,30
10175,10176,30


Отберем 1000 личностей для distractors.

In [9]:
distractors_ids = set(size_df.sample(1000)[1].values)

Теперь создаем датафреймы с фото для query и distractors.

In [10]:
query_df = id_df[id_df[1].isin(query_ids)].groupby(1).sample(3).sort_index().reset_index(drop=True)
query_df

,0,1
0,001628.jpg,1994
1,001929.jpg,2484
2,002276.jpg,7899
3,002951.jpg,245
4,003711.jpg,2275
...,...,...
295,196965.jpg,8062
296,197595.jpg,8188
297,198262.jpg,8244
298,198316.jpg,6038


In [11]:
distractors_df = id_df[id_df[1].isin(distractors_ids)].groupby(1).sample(1).sort_index().reset_index(drop=True)
distractors_df

,0,1
0,000206.jpg,3537
1,000561.jpg,4340
2,000592.jpg,4411
3,000614.jpg,3055
4,000778.jpg,8597
...,...,...
995,202022.jpg,7707
996,202117.jpg,9125
997,202289.jpg,9314
998,202324.jpg,5891


Теперь обрабатываем фото и сохраняем на диск. Группировать по личностям не надо, значит, будет только две папки: query и distractors. В корне будут файлы со списком картинок.

In [12]:
df_to_process = [query_df, distractors_df]
path_to_process = ['metric/query/', 'metric/distractors/']

for path in path_to_process:
    os.makedirs(path, exist_ok=True)

query_df.to_csv('metric/query.csv', index=False, header=False)
distractors_df.to_csv('metric/distractors.csv', index=False, header=False)


In [13]:
mark_df = pd.read_csv('CelebA/Anno/list_landmarks_celeba.txt', header=1, sep='\\s+')

In [14]:
arcface_dst = np.array(
    [[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
     [41.5493, 92.3655], [70.7299, 92.2041]],
    dtype=np.float32) * 2

def get_aligned(image, landmarks):
    dst_landmarks = arcface_dst
    transform = SimilarityTransform.from_estimate(landmarks, dst_landmarks)
    M = transform.params[0:2, :]
    transformed_image = cv2.warpAffine(image, M, (224, 224))
    return transformed_image


In [15]:
for df, path in zip(df_to_process, path_to_process):
    for row in df.itertuples():
        image_name = row._1
        image_path = f'CelebA/Img/img_celeba/{image_name}'
    
        image = np.asarray(Image.open(image_path))
        landmarks = np.asarray(mark_df.loc[image_name]).reshape(-1, 2)

        aligned_image = get_aligned(image, landmarks)

        out_path = f'{path}/{image_name}'
        Image.fromarray(aligned_image).save(out_path)
